# Diet Optimization Project - Staging Environment

## Development Roadmap

### Current Features
- ✅ Multi-restaurant menu items (Chipotle, Subway, McDonald's, Pizza Hut, Taco Bell, etc.)
- ✅ Comprehensive nutrient tracking (17 nutrients: macros + vitamins + minerals)
- ✅ Satisfaction uncertainty modeling via variety constraints
- ✅ Data-driven architecture (all parameters in CSV files)

### Future Enhancements
1. **Stochastic Programming**: Add price/satisfaction scenarios
2. **Sensitivity Analysis**: Shadow prices, what-if scenarios
3. **Visualizations**: Nutrient charts, cost breakdowns, Pareto frontiers
4. **Multi-period Optimization**: Weekly/monthly meal planning


# Diet Optimization Model

## Objective
Minimize total cost while satisfying nutritional requirements, budget constraints, and minimum satisfaction thresholds.

## Key Features
- **17 Nutrients**: Macros (Calories, Protein, Carbs, Fat), vitamins, minerals
- **35 Foods**: Restaurant menu items from 9 chains (Chipotle, Subway, McDonald's, etc.)
- **Budget**: $200-$400
- **Satisfaction**: Minimum threshold with variety constraints (max 3 servings per food)
- **Data-driven**: All parameters loaded from CSV files

## Satisfaction Uncertainty Model
Uses **variety constraints** to model diminishing satisfaction:
- Limits servings per food (prevents over-reliance on favorites)
- Maintains LP structure (solvable with CPLEX)
- Alternative approaches: Stochastic programming, non-linear diminishing returns, piecewise linear approximation

In [15]:
# Import libraries and initialize GAMSPy container
import numpy as np
import pandas as pd
import gamspy as gp
import gamspy.math as gpm
import sys

gp.set_options({'USE_PY_VAR_NAME': 'yes'})
m = gp.Container()

In [16]:
# Define model structure: nutrients, restaurants, and menu items
expanded_nutrients = [
    "Calories", "Protein", "Carbs", "Fat", "SaturatedFat", "TransFat", "Sugars",
    "Sodium", "Fiber", "VitaminA", "VitaminC", "VitaminD", "Calcium", "Iron", 
    "Potassium", "Cholesterol", "Caffeine"
]

restaurants_dict = {
    "Chipotle": ["Chicken_Burrito", "Steak_Bowl", "Veggie_Tacos", "Chicken_Salad"],
    "Subway": ["Turkey_Sandwich", "Veggie_Delite", "Chicken_Teriyaki", "Meatball_Marinara"],
    "McDonalds": ["Big_Mac", "Quarter_Pounder", "Chicken_Nuggets", "French_Fries"],
    "PizzaHut": ["Pepperoni_Pizza", "Cheese_Pizza", "Veggie_Pizza", "Breadsticks"],
    "TacoBell": ["Crunchwrap", "Taco", "Burrito", "Nachos"],
    "Starbucks": ["Latte", "Cappuccino", "Frappuccino", "Muffin", "Croissant"],
    "Dunkin": ["Coffee", "Donut", "Bagel"],
    "DailyScoop": ["Vanilla_Cone", "Chocolate_Sundae", "Strawberry_Scoop", "Cookie_Dough"],
    "ColdStone": ["IceCream_Cake", "Smoothie", "Milkshake"]
}

restaurants_list = list(restaurants_dict.keys())
menu_items_data = [(r, item) for r, items in restaurants_dict.items() for item in items]
food_list = [f"{r}_{item}" for r, items in restaurants_dict.items() for item in items]



In [17]:
# Fetch nutritional data from Nutritionix API
#from IPython.display import HTML, display
#display(HTML('<iframe src="https://www.nutritionix.com/subway/nutrition-calculator" width="100%" height="800"></iframe>'))


In [20]:
# Load nutritional data from CSV
df = pd.read_csv("nutrient_data.csv")

# Create nutrient dictionary for each food
nutrient_values = {}
for _, row in df.iterrows():
    item = row['Restaurant_MenuItem']
    nutrient_values[item] = {
        "Calories": row['Calories'], "Protein": row['Protein'], "Carbs": row['Carbs'], "Fat": row['Fat'],
        "SaturatedFat": row['SaturatedFat'], "TransFat": row['TransFat'], "Sugars": row['Sugars'],
        "Sodium": row['Sodium'], "Fiber": row['Fiber'], "VitaminA": row['VitaminA'], "VitaminC": row['VitaminC'],
        "VitaminD": row['VitaminD'], "Calcium": row['Calcium'], "Iron": row['Iron'], "Potassium": row['Potassium'],
        "Cholesterol": row['Cholesterol'], "Caffeine": row['Caffeine']
    }

# Expand to (food, nutrient, value) tuples for GAMSPy
nutrient_data_expanded = [(food, nutrient, nutrient_values[food].get(nutrient, 0)) 
                          for food in food_list for nutrient in expanded_nutrients]


In [21]:
# Create GAMSPy sets
restaurants = gp.Set(m, name="restaurants", records=restaurants_list)
foods = gp.Set(m, name="foods", records=food_list)
nutrients = gp.Set(m, name="nutrients", records=expanded_nutrients)

# Load food prices from CSV
prices_df = pd.read_csv("food_prices.csv", dtype={'Food': str, 'Restaurant': str})
price_per_serving_data = prices_df[['Food', 'Price']].copy()
price_per_serving_data.columns = ['foods', 'value']
price_per_serving = gp.Parameter(m, name="price_per_serving", domain=[foods], records=price_per_serving_data)

# Load nutritional content per serving
nutrient_per_serving = gp.Parameter(m, name="nutrient_per_serving", domain=[foods, nutrients], records=nutrient_data_expanded)

# Load model configuration parameters
config_df = pd.read_csv("model_config.csv")
config_values = config_df.set_index('Parameter')['Value'].to_dict()

print(f"✅ Loaded: {len(food_list)} foods, {len(expanded_nutrients)} nutrients")


✅ Loaded: 35 foods, 17 nutrients


### Modeling Satisfaction

In [22]:
# Load satisfaction parameters from CSV
satisfaction_df = pd.read_csv("food_satisfaction.csv", dtype={'Food': str})

base_satisfaction_data = satisfaction_df[['Food', 'Base_Satisfaction']].copy()
base_satisfaction_data.columns = ['foods', 'value']
base_satisfaction = gp.Parameter(m, name="base_satisfaction", domain=[foods], records=base_satisfaction_data)

satisfaction_penalty_data = satisfaction_df[['Food', 'Habituation_Rate']].copy()
satisfaction_penalty_data.columns = ['foods', 'value']
satisfaction_penalty = gp.Parameter(m, name="satisfaction_penalty", domain=[foods], records=satisfaction_penalty_data)

min_satisfaction = gp.Parameter(m, name="min_satisfaction", records=float(config_values['min_satisfaction']))

In [7]:
# Load nutrient constraints (min/max bounds) from CSV
constraints_df = pd.read_csv("nutrient_constraints.csv", dtype={'Nutrient': str})

Nmin_data = constraints_df[['Nutrient', 'Nmin']].copy()
Nmin_data.columns = ['nutrients', 'value']
Nmin = gp.Parameter(m, name="Nmin", domain=[nutrients], records=Nmin_data)

Nmax_data = constraints_df[['Nutrient', 'Nmax']].copy()
Nmax_data.columns = ['nutrients', 'value']
Nmax = gp.Parameter(m, name="Nmax", domain=[nutrients], records=Nmax_data)


In [8]:
# Extract model parameters from config
w_cost = gp.Parameter(m, name="w_cost", records=float(config_values['w_cost']))
w_satisfaction = gp.Parameter(m, name="w_satisfaction", records=float(config_values['w_satisfaction']))
max_servings_per_food = gp.Parameter(m, name="max_servings_per_food", records=float(config_values['max_servings_per_food']))

budget_min = float(config_values['budget_min'])
budget_max = float(config_values['budget_max'])
variety_bonus_weight = float(config_values['variety_bonus_weight'])

In [9]:
# Load food serving bounds from CSV file
# CSV Format: Food,Fmin,Fmax,Description
bounds_csv = "food_bounds.csv"
bounds_df = pd.read_csv(bounds_csv, dtype={'Food': str})

# Create DataFrames for Fmin and Fmax (column names must match domain set name)
Fmin_data = bounds_df[['Food', 'Fmin']].copy()
Fmin_data.columns = ['foods', 'value']  # Column name 'foods' matches the set name

Fmax_data = bounds_df[['Food', 'Fmax']].copy()
Fmax_data.columns = ['foods', 'value']  # Column name 'foods' matches the set name

# Fmini = minimum number of required servings of food i, ∀i∈F
Fmin = gp.Parameter(m, name="Fmin", domain=[foods], records=Fmin_data)

# Fmaxi = maximum allowable number of servings of food i, ∀i∈F
Fmax = gp.Parameter(m, name="Fmax", domain=[foods], records=Fmax_data)

# Nutrient level constraints: Nminj and Nmaxj
# Nminj = minimum required level of nutrient j, ∀j∈N
Nmin = gp.Parameter(m, name="Nmin", domain=[nutrients],
                    records=pd.Series({
                        "Calories": 1800,   # Minimum calories per day
                        "Protein": 70,      # Minimum grams of protein
                        "Carbs": 400,       # Minimum grams of carbs
                        "Fat": 200           # Minimum grams of fat
                    }))

# Nmaxj = maximum allowable level of nutrient j, ∀j∈N
Nmax = gp.Parameter(m, name="Nmax", domain=[nutrients],
                    records=pd.Series({
                        "Calories": 25000,   # Maximum calories per day
                        "Protein": 1500,     # Maximum grams of protein
                        "Carbs": 5000,       # Maximum grams of carbs
                        "Fat": 30000           # Maximum grams of fat
                    }))

In [10]:
# Variables
# xi = number of servings of food i to purchase/consume, ∀i∈F
x = gp.Variable(m, name="x", domain=[foods], type="positive", description="number of servings of food i")

# Set bounds using Fmin and Fmax parameters
x.lo[foods] = Fmin[foods]  # Lower bound: minimum servings
x.up[foods] = Fmax[foods]  # Upper bound: maximum servings

In [11]:
# Equations (Constraints)

# Constraint Set 1: For each nutrient j∈N, at least meet the minimum required level
# ∑(i∈F) aij*xi ≥ Nminj, ∀j∈N
nutrient_min = gp.Equation(m, name="nutrient_min", domain=[nutrients], description="minimum nutrient requirements")
nutrient_min[nutrients] = gp.Sum(foods, nutrient_per_serving[foods, nutrients] * x[foods]) >= Nmin[nutrients]

# Constraint Set 2: For each nutrient j∈N, do not exceed the maximum allowable level
# ∑(i∈F) aij*xi ≤ Nmaxj, ∀j∈N
nutrient_max = gp.Equation(m, name="nutrient_max", domain=[nutrients], description="maximum nutrient limits")
nutrient_max[nutrients] = gp.Sum(foods, nutrient_per_serving[foods, nutrients] * x[foods]) <= Nmax[nutrients]

# Constraint: Total cost (budget) must be within configured limits
cost_min = gp.Equation(m, name="cost_min", description=f"minimum budget constraint (${budget_min})")
cost_min[:] = gp.Sum(foods, price_per_serving[foods] * x[foods]) >= budget_min

cost_max = gp.Equation(m, name="cost_max", description=f"maximum budget constraint (${budget_max})")
cost_max[:] = gp.Sum(foods, price_per_serving[foods] * x[foods]) <= budget_max
# Note: Constraint Set 3 (xi ≥ Fmini) and Constraint Set 4 (xi ≤ Fmaxi) 
# are already handled by the variable bounds set in Cell 4

# Constraint: Variety - limit servings of any single food to encourage diversity
# This addresses diminishing satisfaction by preventing over-consumption of favorites
variety_constraint = gp.Equation(m, name="variety_constraint", domain=[foods],
                                description="Maximum servings per food for variety")
variety_constraint[foods] = x[foods] <= max_servings_per_food

# Constraint: Total satisfaction must be above minimum threshold
# ∑(i∈F) base_satisfaction[i] * x[i] ≥ min_satisfaction
satisfaction_constraint = gp.Equation(m, name="satisfaction_constraint", description="Minimum satisfaction requirement")
satisfaction_constraint[:] = gp.Sum(foods, base_satisfaction[foods] * x[foods]) >= min_satisfaction

In [12]:
# Objective Function: Cost minimization with variety constraints
# Minimize: total_cost
# Satisfaction and variety are handled through constraints

total_cost = gp.Sum(foods, price_per_serving[foods] * x[foods])
total_base_satisfaction = gp.Sum(foods, base_satisfaction[foods] * x[foods])

# Objective: minimize total cost (satisfaction is handled as constraint)
obj_expr = total_cost

In [13]:


# Create and solve the model
diet_plan = gp.Model(
    m,
    equations=m.getEquations(),
    problem=gp.Problem.LP,
    sense=gp.Sense.MIN,  # Minimize total cost
    objective=obj_expr,
    name="diet_plan",
)

# Solve the model
diet_plan.solve(output=sys.stdout)



--- Job _da4By2uCTCeZ5W4FzTcGlw.gms Start 12/13/25 00:43:23 52.1.0 4f802a74 WEX-WEI x86 64bit/MS Windows
--- Applying:
    C:\Users\Shashwat\Desktop\CS 524 Introduction to optimization\.venv\Lib\site-packages\gamspy_base\gmsprmNT.txt
--- GAMS Parameters defined
    LP cplex
    Input C:\Users\Shashwat\AppData\Local\Temp\tmpyf_3ij5x\_da4By2uCTCeZ5W4FzTcGlw.gms
    Output C:\Users\Shashwat\AppData\Local\Temp\tmpyf_3ij5x\_da4By2uCTCeZ5W4FzTcGlw.lst
    ScrDir C:\Users\Shashwat\AppData\Local\Temp\tmpyf_3ij5x\tmpdorlxh50\
    SysDir "C:\Users\Shashwat\Desktop\CS 524 Introduction to optimization\.venv\Lib\site-packages\gamspy_base\"
    LogOption 3
    Trace C:\Users\Shashwat\AppData\Local\Temp\tmpyf_3ij5x\_da4By2uCTCeZ5W4FzTcGlw.txt
    License C:\Users\Shashwat\Documents\GAMSPy\gamspy_license.txt
    OptFile 0
    OptDir C:\Users\Shashwat\AppData\Local\Temp\tmpyf_3ij5x\
    LimRow 0
    LimCol 0
    TraceOpt 3
    GDX C:\Users\Shashwat\AppData\Local\Temp\tmpyf_3ij5x\_da4By2uCTCeZ5W4FzTcGlw

,Solver Status,Model Status,Objective,Num of Equations,Num of Variables,Model Type,Solver,Solver Time
0,Normal,InfeasibleNoSolution,NaN,73,36,LP,CPLEX,0.0


In [14]:
# Display results
print("\n=== OPTIMIZATION RESULTS ===")
print(f"Objective Function Value (Total Cost): ${diet_plan.objective_value:.2f}")
print(f"\nSolution Status: {diet_plan.status}")
print(f"Solver Status: {diet_plan.solve_status}")

print("\n=== FOOD SERVINGS ===")
print(x.records)

print("\n=== SOLUTION ANALYSIS ===")

# Get solution values
food_col = x.records.columns[0]
x_values = {}
foods_used = 0
for idx, row in x.records.iterrows():
    food_name = row[food_col]
    servings = row['level']
    x_values[food_name] = servings
    if servings > 0:
        foods_used += 1

print(f"🍽️  Variety Analysis:")
print(f"   Foods used: {foods_used} out of {len(foods.toList())} available")
print(f"   Max servings per food: {max_servings_per_food.toList()[0]:.1f}")

# Calculate satisfaction metrics
total_satisfaction = sum(base_satisfaction[food].toList()[0] * servings
                        for food, servings in x_values.items() if servings > 0)
avg_satisfaction_per_serving = total_satisfaction / sum(x_values.values()) if sum(x_values.values()) > 0 else 0

print(f"\n😊 Satisfaction Analysis:")
print(f"   Total satisfaction: {total_satisfaction:.1f} (Min required: {min_satisfaction.toList()[0]:.1f})")
print(f"   Average satisfaction per serving: {avg_satisfaction_per_serving:.2f}")

print(f"\n=== NUTRIENT TOTALS ===")
# Calculate nutrient totals
nutrient_totals = {}
for n in nutrients.toList():
    total = 0.0
    # Get nutrient values from parameter records
    for idx, row in nutrient_per_serving.records.iterrows():
        # Check if this row is for the current nutrient
        nutrient_name = row[nutrients.name] if nutrients.name in row.index else row.iloc[1]
        food_name = row[foods.name] if foods.name in row.index else row.iloc[0]

        if nutrient_name == n and food_name in x_values:
            nutrient_amt = row['value']
            servings = x_values[food_name]
            total += nutrient_amt * servings

    nutrient_totals[n] = total
    print(f"{n}: {total:.2f} (Min: {Nmin[n]}, Max: {Nmax[n]})")

print(f"\n📋 Satisfaction Uncertainty Modeling:")
print(f"   - Variety enforced: Max {max_servings_per_food.toList()[0]:.0f} servings per food")
print(f"   - This prevents over-reliance on favorites (diminishing satisfaction)")
print(f"   - Encourages dietary diversity for sustained satisfaction")


=== OPTIMIZATION RESULTS ===
Objective Function Value (Total Cost): $2000000000000000105009520510408840497408937162216318309831708231023604915977816391572742750160895728087408887665767756353885046470720861151289584369573413965696774401853151607475660467589576180118737906469941599890162238077935281760149305485560284989158517577640113685676231338944392773730918801080320.00

Solution Status: ModelStatus.InfeasibleNoSolution
Solver Status: SolveStatus.NormalCompletion

=== FOOD SERVINGS ===
                          foods  level  marginal  lower  upper  scale
0      Chipotle_Chicken_Burrito    0.0       0.0    0.0    3.0    1.0
1           Chipotle_Steak_Bowl    0.0       0.0    0.0    3.0    1.0
2         Chipotle_Veggie_Tacos    0.0       0.0    0.0    3.0    1.0
3        Chipotle_Chicken_Salad    0.0       0.0    0.0    3.0    1.0
4        Subway_Turkey_Sandwich    0.0       0.0    0.0    3.0    1.0
5          Subway_Veggie_Delite    0.0       0.0    0.0    3.0    1.0
6       Subway_Ch

Protein: 0.00 (Min: ImplicitParameter(parent=Parameter(name='Nmin', domain=[Set(name='nutrients', domain=['*'])]), name='Nmin', domain=[], permutation=None), parent_scalar_domains=[]), Max: ImplicitParameter(parent=Parameter(name='Nmax', domain=[Set(name='nutrients', domain=['*'])]), name='Nmax', domain=[], permutation=None), parent_scalar_domains=[]))
Carbs: 0.00 (Min: ImplicitParameter(parent=Parameter(name='Nmin', domain=[Set(name='nutrients', domain=['*'])]), name='Nmin', domain=[], permutation=None), parent_scalar_domains=[]), Max: ImplicitParameter(parent=Parameter(name='Nmax', domain=[Set(name='nutrients', domain=['*'])]), name='Nmax', domain=[], permutation=None), parent_scalar_domains=[]))
Fat: 0.00 (Min: ImplicitParameter(parent=Parameter(name='Nmin', domain=[Set(name='nutrients', domain=['*'])]), name='Nmin', domain=[], permutation=None), parent_scalar_domains=[]), Max: ImplicitParameter(parent=Parameter(name='Nmax', domain=[Set(name='nutrients', domain=['*'])]), name='Nmax'